In [72]:
import pandas as pd
import numpy as np

# ---------- 1) Load data ----------
unity_path = r"C:\Users\Marcel\Desktop\Study Project\VR Data\UnityObjectsSizes.csv"
eyetrack_path = r"C:\Users\Marcel\Desktop\Study Project\VR Data\Data with Turns\combined_dataframe.csv"
et = pd.read_csv(eyetrack_path)
unity = pd.read_csv(unity_path)

In [73]:
# ---------- Define key columns ----------
ET_NAME_COL = "names"
ET_COLLIDER_COL = "hitColliderType"

UNITY_NAME_COL = "GameObject"
UNITY_COLLIDER_COL = "ColliderType"

# Unity geometry columns to attach
UNITY_GEOM_COLS = ["CenterX", "CenterY", "CenterZ", "SizeX", "SizeY", "SizeZ"]

EVENTS_COL = "events"
LONG_EVENTS_COL = "long_events"

SUBJ_COL = "SubjectID"

In [74]:
# ---------- Normalize object names and collider types ----------
def normalize_obj_name(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip()

def normalize_collider_et(s: pd.Series) -> pd.Series:
    # "UnityEngine.MeshCollider" -> "MeshCollider"
    return (
        s.astype(str)
         .str.strip()
         .str.replace("UnityEngine.", "", regex=False)
    )

def normalize_collider_unity(s: pd.Series) -> pd.Series:
    # already "MeshCollider", "BoxCollider", etc.
    return s.astype(str).str.strip()

et["obj_name_norm"] = normalize_obj_name(et[ET_NAME_COL])
et["collider_type_norm"] = normalize_collider_et(et[ET_COLLIDER_COL])

unity["obj_name_norm"] = normalize_obj_name(unity[UNITY_NAME_COL])
unity["collider_type_norm"] = normalize_collider_unity(unity[UNITY_COLLIDER_COL])

In [75]:
# ---------- Count Unity multiplicity AND check whether duplicate pairs differ in size,
#            and compute size ranges within each (name, collider) pair ----------
unity_pair_size_stats = (
    unity
    .groupby(["obj_name_norm", "collider_type_norm"], dropna=False)
    .agg(
        unity_n=("obj_name_norm", "size"),

        sizex_nuniq=("SizeX", "nunique"),
        sizey_nuniq=("SizeY", "nunique"),
        sizez_nuniq=("SizeZ", "nunique"),

        sizex_min=("SizeX", "min"),
        sizex_max=("SizeX", "max"),
        sizey_min=("SizeY", "min"),
        sizey_max=("SizeY", "max"),
        sizez_min=("SizeZ", "min"),
        sizez_max=("SizeZ", "max"),
    )
    .reset_index()
)

# Compute ranges (max - min)
unity_pair_size_stats["sizex_range"] = unity_pair_size_stats["sizex_max"] - unity_pair_size_stats["sizex_min"]
unity_pair_size_stats["sizey_range"] = unity_pair_size_stats["sizey_max"] - unity_pair_size_stats["sizey_min"]
unity_pair_size_stats["sizez_range"] = unity_pair_size_stats["sizez_max"] - unity_pair_size_stats["sizez_min"]

# A pair is only "problematic" if it appears >1 times AND at least one size dimension varies
unity_pair_size_stats["size_varies"] = (
    unity_pair_size_stats[["sizex_nuniq", "sizey_nuniq", "sizez_nuniq"]].gt(1).any(axis=1)
)

total_pairs = len(unity_pair_size_stats)
unique_pairs = (unity_pair_size_stats["unity_n"] == 1).sum()
dup_pairs_total = (unity_pair_size_stats["unity_n"] > 1).sum()

dup_pairs_safe_same_size = (
    (unity_pair_size_stats["unity_n"] > 1) &
    (~unity_pair_size_stats["size_varies"])
).sum()

dup_pairs_problematic = (
    (unity_pair_size_stats["unity_n"] > 1) &
    (unity_pair_size_stats["size_varies"])
).sum()

print("---- Unity pair multiplicity with size check ----")
print(f"Distinct (name, collider) pairs in Unity: {total_pairs}")
print(f"Pairs occurring exactly once (unique): {unique_pairs}")
print(f"Pairs occurring >1 times (duplicates total): {dup_pairs_total}")
print(f"Duplicates with identical size (treat as safe): {dup_pairs_safe_same_size}")
print(f"Duplicates with differing size (problematic): {dup_pairs_problematic}")

# ---------- List only the problematic duplicates, including min/max/range ----------
problematic_unity_pairs = (
    unity_pair_size_stats
    .loc[(unity_pair_size_stats["unity_n"] > 1) & (unity_pair_size_stats["size_varies"])]
    .sort_values("unity_n", ascending=False)
)

print("\n---- Problematic Unity objects (same name + collider type, different size) ----")
print(
    problematic_unity_pairs[
        [
            "obj_name_norm", "collider_type_norm", "unity_n",

            "sizex_nuniq", "sizex_min", "sizex_max", "sizex_range",
            "sizey_nuniq", "sizey_min", "sizey_max", "sizey_range",
            "sizez_nuniq", "sizez_min", "sizez_max", "sizez_range",
        ]
    ]
)

---- Unity pair multiplicity with size check ----
Distinct (name, collider) pairs in Unity: 1451
Pairs occurring exactly once (unique): 1312
Pairs occurring >1 times (duplicates total): 139
Duplicates with identical size (treat as safe): 30
Duplicates with differing size (problematic): 109

---- Problematic Unity objects (same name + collider type, different size) ----
         obj_name_norm collider_type_norm  unity_n  sizex_nuniq  sizex_min  \
645   CollisionObject1     SphereCollider      357           26   3.643121   
646   CollisionObject2     SphereCollider      357           24   2.687710   
644   CollisionObject0    CapsuleCollider      357           64   0.566223   
1291           Wheel04        BoxCollider      273          253   0.205872   
1288           Wheel01        BoxCollider      273          253   0.205872   
...                ...                ...      ...          ...        ...   
1246        Trash_v1_2        BoxCollider        2            2   0.924377   
1172

In [76]:
# ---------- Build set of Unity (name, collider) pairs that are safe to merge ----------
safe_unity_pairs = unity_pair_size_stats.loc[
    (unity_pair_size_stats["unity_n"] == 1) |
    (
        (unity_pair_size_stats["unity_n"] > 1) &
        (~unity_pair_size_stats["size_varies"])
    ),
    ["obj_name_norm", "collider_type_norm"]
].copy()

# Turn into a set of tuples for fast lookup
safe_unity_pair_set = set(
    zip(safe_unity_pairs["obj_name_norm"], safe_unity_pairs["collider_type_norm"])
)

print(f"Safe Unity pairs (unique or duplicate same-size): {len(safe_unity_pair_set)}")

Safe Unity pairs (unique or duplicate same-size): 1342


In [77]:
# ---------------------------------------------------------
# Create fixation_id inside et (keep full et; non-fixation rows get <NA>)
# ---------------------------------------------------------
et[EVENTS_COL] = pd.to_numeric(et[EVENTS_COL], errors="coerce")

et["fixation_id"] = pd.NA
fix_id_counter = 0

for subj, g in et.groupby(SUBJ_COL, sort=False):
    valid_onsets = g.index[(g[EVENTS_COL] == 2) & (g[LONG_EVENTS_COL].notna())].to_list()

    for onset_idx in valid_onsets:
        onset_pos = g.index.get_loc(onset_idx)

        offset_pos = None
        for pos in range(onset_pos, len(g)):
            if g.iloc[pos][EVENTS_COL] == -2:
                offset_pos = pos
                break

        if offset_pos is None:
            continue

        fix_id_counter += 1
        seg_index = g.index[onset_pos:offset_pos + 1]
        et.loc[seg_index, "fixation_id"] = fix_id_counter

print(f"Total fixations detected (valid onset..offset): {fix_id_counter}")

fix_rows = et["fixation_id"].notna()

Total fixations detected (valid onset..offset): 147066


In [78]:
# ---------------------------------------------------------
# Normalize collider type within each fixation by majority vote with tie-break
# Mesh > Box > Sphere > Capsule
# ---------------------------------------------------------
COLLIDER_PRIORITY = ["MeshCollider", "BoxCollider", "SphereCollider", "CapsuleCollider"]

def choose_collider_type(series: pd.Series) -> str:
    s = series.dropna().astype(str)
    if s.empty:
        return np.nan
    counts = s.value_counts()
    max_count = counts.max()
    tied = set(counts[counts == max_count].index.tolist())

    for c in COLLIDER_PRIORITY:
        if c in tied:
            return c

    return sorted(tied)[0]

chosen_collider_per_fix = (
    et.loc[fix_rows]
      .groupby("fixation_id")["collider_type_norm"]
      .apply(choose_collider_type)
)

# broadcast chosen collider back to ALL rows in each fixation
et.loc[fix_rows, "collider_type_norm_fix"] = (
    et.loc[fix_rows, "fixation_id"].map(chosen_collider_per_fix)
)

In [79]:
# ---------------------------------------------------------
# Build fixation-level (obj_name_norm, chosen_collider) pairs
# (obj_name_norm assumed constant within fixation; take first)
# ---------------------------------------------------------
fix_pairs = (
    et.loc[fix_rows, ["fixation_id", "obj_name_norm", "collider_type_norm_fix"]]
      .groupby("fixation_id", as_index=False)
      .agg(
          obj_name_norm=("obj_name_norm", "first"),
          collider_type_norm=("collider_type_norm_fix", "first"),
      )
)

# ---------------------------------------------------------
# Safe Unity lookup (safe pairs only), pick one row per pair deterministically
# ---------------------------------------------------------
safe_unity_df = unity.merge(
    safe_unity_pairs,
    on=["obj_name_norm", "collider_type_norm"],
    how="inner"
)

unity_safe_lookup = (
    safe_unity_df
    .sort_values(["obj_name_norm", "collider_type_norm"])
    .groupby(["obj_name_norm", "collider_type_norm"], as_index=False)
    .first()[["obj_name_norm", "collider_type_norm"] + UNITY_GEOM_COLS]
)

# ---------------------------------------------------------
# Match fixations to safe pairs + merge geometry back to fixation rows only
# ---------------------------------------------------------
fix_pairs["is_safe_match"] = list(zip(fix_pairs["obj_name_norm"], fix_pairs["collider_type_norm"]))
fix_pairs["is_safe_match"] = fix_pairs["is_safe_match"].apply(lambda t: t in safe_unity_pair_set)

fix_pairs_geom = fix_pairs.merge(
    unity_safe_lookup,
    on=["obj_name_norm", "collider_type_norm"],
    how="left"
)

# merge fixation-level geometry onto full et by fixation_id
et = et.merge(
    fix_pairs_geom[["fixation_id", "is_safe_match"] + UNITY_GEOM_COLS],
    on="fixation_id",
    how="left"
)

# ---------------------------------------------------------
# Counts + print unmatched pairs for inspection
# ---------------------------------------------------------
n_fix_total = len(fix_pairs)
n_fix_matched = int(fix_pairs["is_safe_match"].sum())
n_fix_unmatched = int((~fix_pairs["is_safe_match"]).sum())

print("---- Fixation → safe Unity match counts ----")
print(f"Fixations (valid onset..offset): {n_fix_total}")
print(f"Fixations with safe Unity pair match: {n_fix_matched}")
print(f"Fixations with NO safe Unity pair match: {n_fix_unmatched}")

unmatched_fix_pairs = (
    fix_pairs.loc[~fix_pairs["is_safe_match"], ["obj_name_norm", "collider_type_norm"]]
    .drop_duplicates()
    .sort_values(["obj_name_norm", "collider_type_norm"])
)

print("\n---- Unique unmatched fixation (obj_name_norm, collider_type_norm) pairs ----")
print(unmatched_fix_pairs)

---- Fixation → safe Unity match counts ----
Fixations (valid onset..offset): 147066
Fixations with safe Unity pair match: 107898
Fixations with NO safe Unity pair match: 39168

---- Unique unmatched fixation (obj_name_norm, collider_type_norm) pairs ----
                        obj_name_norm collider_type_norm
34290                          01_Cma       MeshCollider
135202                         02_Cma       MeshCollider
68594                          03_Cma       MeshCollider
53552                          06_Cma       MeshCollider
9013                           07_Cma       MeshCollider
...                               ...                ...
135658                  terrain_O.001        BoxCollider
139514                  terrain_R.001        BoxCollider
325        the-oval-cork-pub_collider       MeshCollider
72616       traffic_light_single_tall       MeshCollider
2323    troyes-shop-7-france_collider       MeshCollider

[408 rows x 2 columns]


In [80]:
# ---------------------------------------------------------
# Resolve unmatched fixations using hitpoint proximity to Unity Center*
# Behavior:
#   1) Try best match among Unity candidates with (name + collider)
#      - if best is within cutoff -> accept as "name+collider"
#      - if best exists but is NOT within cutoff -> still try name-only
#   2) If no (name + collider) candidates exist -> try name-only directly
#   3) Accept only if the chosen best match is within DIST_CUTOFF
#   4) End report: which fixations still have no accepted match
# ---------------------------------------------------------

HPX, HPY, HPZ = "hitPointOnObject_x", "hitPointOnObject_y", "hitPointOnObject_z"
DIST_CUTOFF = 10.0  # Unity units

# --- numeric coercion ---
for c in [HPX, HPY, HPZ]:
    et[c] = pd.to_numeric(et[c], errors="coerce")
for c in ["CenterX", "CenterY", "CenterZ"]:
    unity[c] = pd.to_numeric(unity[c], errors="coerce")

# --- unmatched fixations (no safe match) ---
fix_rows = et["fixation_id"].notna()
unmatched_fix_ids = (
    et.loc[fix_rows & (et["is_safe_match"] == False), "fixation_id"]
      .dropna()
      .unique()
)

print(f"Unmatched fixations to try resolving: {len(unmatched_fix_ids)}")

# --- fixation-level summary: mean hitpoint + fixation pair (name, normalized collider) ---
fix_sum = (
    et.loc[et["fixation_id"].isin(unmatched_fix_ids),
           ["fixation_id", "obj_name_norm", "collider_type_norm_fix", HPX, HPY, HPZ]]
      .groupby("fixation_id", as_index=False)
      .agg(
          obj_name_norm=("obj_name_norm", "first"),
          collider_type_norm=("collider_type_norm_fix", "first"),
          hx=(HPX, "mean"),
          hy=(HPY, "mean"),
          hz=(HPZ, "mean"),
          hp_n=(HPX, lambda x: x.notna().sum())
      )
      .dropna(subset=["hx", "hy", "hz"])
)

unity_cand = unity.dropna(subset=["obj_name_norm", "CenterX", "CenterY", "CenterZ"]).copy()

def best_match(cand_df: pd.DataFrame, hx: float, hy: float, hz: float):
    if len(cand_df) == 0:
        return None
    dx = cand_df["CenterX"].to_numpy() - hx
    dy = cand_df["CenterY"].to_numpy() - hy
    dz = cand_df["CenterZ"].to_numpy() - hz
    d = np.sqrt(dx*dx + dy*dy + dz*dz)
    i = int(np.argmin(d))
    return cand_df.iloc[i], float(d[i])

results = []

for _, r in fix_sum.iterrows():
    fix_id = r["fixation_id"]
    name = r["obj_name_norm"]
    col  = r["collider_type_norm"]
    hx, hy, hz = float(r["hx"]), float(r["hy"]), float(r["hz"])
    hp_n = int(r["hp_n"])

    # Candidate sets
    cand_nc = unity_cand[
        (unity_cand["obj_name_norm"] == name) &
        (unity_cand["collider_type_norm"] == col)
    ]
    cand_n = unity_cand[unity_cand["obj_name_norm"] == name]

    # Try name+collider first
    m_nc = best_match(cand_nc, hx, hy, hz)

    best_row = None
    best_dist = np.nan
    strategy = "none"

    if m_nc is not None:
        row_nc, dist_nc = m_nc

        if dist_nc <= DIST_CUTOFF:
            best_row, best_dist = row_nc, dist_nc
            strategy = "name+collider"
        else:
            # name+collider exists but too far -> still try name-only
            m_n = best_match(cand_n, hx, hy, hz)
            if m_n is not None:
                row_n, dist_n = m_n
                if dist_n <= DIST_CUTOFF:
                    best_row, best_dist = row_n, dist_n
                    strategy = "name_only"
                else:
                    # no accepted match; keep closest attempt for diagnostics
                    # prefer the smaller of the two distances
                    if dist_n < dist_nc:
                        best_row, best_dist = row_n, dist_n
                        strategy = "name_only_too_far"
                    else:
                        best_row, best_dist = row_nc, dist_nc
                        strategy = "name+collider_too_far"
            else:
                best_row, best_dist = row_nc, dist_nc
                strategy = "name+collider_too_far"

    else:
        # no name+collider candidates -> try name-only
        m_n = best_match(cand_n, hx, hy, hz)
        if m_n is not None:
            row_n, dist_n = m_n
            best_row, best_dist = row_n, dist_n
            strategy = "name_only" if dist_n <= DIST_CUTOFF else "name_only_too_far"
        else:
            best_row, best_dist = None, np.nan
            strategy = "none"

    accepted = strategy in ["name+collider", "name_only"]

    out = {
        "fixation_id": fix_id,
        "obj_name_norm": name,
        "collider_type_norm_fix": col,
        "hp_n": hp_n,
        "strategy": strategy,
        "best_dist": best_dist,
        "accepted": accepted,
    }

    if best_row is not None:
        out.update({
            "unity_collider_type_norm": best_row["collider_type_norm"],
            "CenterX_match": best_row["CenterX"],
            "CenterY_match": best_row["CenterY"],
            "CenterZ_match": best_row["CenterZ"],
            "SizeX_match": best_row["SizeX"],
            "SizeY_match": best_row["SizeY"],
            "SizeZ_match": best_row["SizeZ"],
            "candidates_name+collider": int(len(cand_nc)),
            "candidates_name_only": int(len(cand_n)),
        })
    else:
        out.update({
            "candidates_name+collider": int(len(cand_nc)),
            "candidates_name_only": int(len(cand_n)),
        })

    results.append(out)

results_df = pd.DataFrame(results)

print("\n---- Proximity matching summary ----")
print("Tried fixations:", len(results_df))
print("Accepted matches (<= cutoff):", int(results_df["accepted"].sum()))
print("Not accepted:", int((~results_df["accepted"]).sum()))
print("\nStrategy counts:")
print(results_df["strategy"].value_counts(dropna=False))

# --- overview of fixations that still have no accepted match ---
still_no_match = results_df[~results_df["accepted"]].copy().sort_values(["strategy", "best_dist"])
print("\n---- Fixations still without an accepted match (first 50) ----")
print(still_no_match.head(50)[
    ["fixation_id", "obj_name_norm", "collider_type_norm_fix", "strategy", "best_dist",
     "candidates_name+collider", "candidates_name_only", "hp_n"]
])


Unmatched fixations to try resolving: 39168

---- Proximity matching summary ----
Tried fixations: 39168
Accepted matches (<= cutoff): 26161
Not accepted: 13007

Strategy counts:
strategy
name+collider            25430
name+collider_too_far     8462
none                      3830
name_only                  731
name_only_too_far          715
Name: count, dtype: int64

---- Fixations still without an accepted match (first 50) ----
       fixation_id           obj_name_norm collider_type_norm_fix  \
14527        56046                 Fence_5           MeshCollider   
10692        39792        CollisionObject1         SphereCollider   
38664       145123  Euro_v6_1_Forged_fence           MeshCollider   
8258         29722                    body            BoxCollider   
18842        72397        CollisionObject0        CapsuleCollider   
875           3358                    Body            BoxCollider   
39137       146872            Building_116           MeshCollider   
15914        60

In [81]:
import re

# Build lookup from accepted proximity matches
accepted_matches = results_df[results_df["accepted"]].copy()

accepted_geom = accepted_matches[[
    "fixation_id",
    "CenterX_match", "CenterY_match", "CenterZ_match",
    "SizeX_match", "SizeY_match", "SizeZ_match"
]].copy()

# Merge helper columns onto et
et = et.merge(accepted_geom, on="fixation_id", how="left")

fix_rows = et["fixation_id"].notna()
fill_cond = fix_rows & (et["is_safe_match"] == False) & (et["SizeX_match"].notna())

# Track what was missing BEFORE filling
sizex_was_na = et["SizeX"].isna()
sizey_was_na = et["SizeY"].isna()
sizez_was_na = et["SizeZ"].isna()

# Fill missing values only (do not overwrite safe-merged values)
for base, mcol in [
    ("CenterX", "CenterX_match"), ("CenterY", "CenterY_match"), ("CenterZ", "CenterZ_match"),
    ("SizeX", "SizeX_match"), ("SizeY", "SizeY_match"), ("SizeZ", "SizeZ_match")
]:
    et.loc[fill_cond & et[base].isna(), base] = et.loc[fill_cond & et[base].isna(), mcol]

# Count actual fills (NaN -> non-NaN) for each dimension
filled_sizex = int((fill_cond & sizex_was_na & et["SizeX"].notna()).sum())
filled_sizey = int((fill_cond & sizey_was_na & et["SizeY"].notna()).sum())
filled_sizez = int((fill_cond & sizez_was_na & et["SizeZ"].notna()).sum())

print("---- Proximity merge into ET ----")
print(f"Fixation rows eligible (unmatched safe-pair + accepted proximity): {int(fill_cond.sum())}")
print(f"Rows actually filled: SizeX {filled_sizex}, SizeY {filled_sizey}, SizeZ {filled_sizez}")

# Optional: drop helper columns after filling
et.drop(columns=["CenterX_match","CenterY_match","CenterZ_match","SizeX_match","SizeY_match","SizeZ_match"], inplace=True)

---- Proximity merge into ET ----
Fixation rows eligible (unmatched safe-pair + accepted proximity): 317170
Rows actually filled: SizeX 317170, SizeY 317170, SizeZ 317170


In [82]:
# =========================================================
# PART 2) For fixations still without accepted match:
#         compute Unity mean+SD size by BASE NAME (strip trailing ' (n)')
# =========================================================

# Fixation IDs that are still unmatched after proximity
still_no_match_fix_ids = still_no_match["fixation_id"].dropna().unique()

# Their names (one per fixation)
still_fix_names = (
    et.loc[et["fixation_id"].isin(still_no_match_fix_ids), ["fixation_id", "obj_name_norm"]]
      .groupby("fixation_id", as_index=False)
      .agg(obj_name_norm=("obj_name_norm", "first"))
)

def strip_trailing_paren_number(name: str) -> str:
    return re.sub(r"\s*\(\d+\)\s*$", "", str(name)).strip()

still_fix_names["obj_base"] = still_fix_names["obj_name_norm"].apply(strip_trailing_paren_number)

# Make the same base-name column in unity
unity["obj_base"] = unity["obj_name_norm"].apply(strip_trailing_paren_number)

# Ensure Unity sizes are numeric
for c in ["SizeX", "SizeY", "SizeZ"]:
    unity[c] = pd.to_numeric(unity[c], errors="coerce")

# Size stats across all Unity rows for each base name
unity_size_stats_by_base = (
    unity.dropna(subset=["obj_base"])
         .groupby("obj_base", as_index=False)
         .agg(
             unity_rows=("obj_base", "size"),
             sizex_mean=("SizeX", "mean"),
             sizex_sd=("SizeX", "std"),
             sizey_mean=("SizeY", "mean"),
             sizey_sd=("SizeY", "std"),
             sizez_mean=("SizeZ", "mean"),
             sizez_sd=("SizeZ", "std"),
         )
)

# Report only base names that appear among still-unmatched fixations
unmatched_base_names = still_fix_names[["obj_base"]].drop_duplicates()
unmatched_unity_size_report = unmatched_base_names.merge(
    unity_size_stats_by_base,
    on="obj_base",
    how="left"
).sort_values("unity_rows", ascending=False)

not_found_in_unity = unmatched_unity_size_report[unmatched_unity_size_report["unity_rows"].isna()].copy()

print("\n---- Unity size variance for still-unmatched fixation names (base-stripped) ----")
print(f"Still-unmatched fixations: {len(still_no_match_fix_ids)}")
print(f"Unique base names among still-unmatched: {len(unmatched_unity_size_report)}")
print(f"Base names not found in Unity: {len(not_found_in_unity)}")

print("\nTop 50 size-variance rows:")
print(unmatched_unity_size_report.head(50))

print("\nBase names not found in Unity (first 50):")
print(not_found_in_unity.head(50)[["obj_base"]])


---- Unity size variance for still-unmatched fixation names (base-stripped) ----
Still-unmatched fixations: 13007
Unique base names among still-unmatched: 151
Base names not found in Unity: 20

Top 50 size-variance rows:
                        obj_base  unity_rows  sizex_mean      sizex_sd  \
21         scaffold_base_02_LOD0      1090.0    1.147497  2.299230e+00   
130    scaffold_topboard_01_LOD0       436.0    2.781368  2.790462e+00   
1               CollisionObject1       357.0    8.475463  6.842456e-01   
13              CollisionObject0       357.0    1.492473  7.375727e-01   
3               CollisionObject2       357.0    6.252766  5.048023e-01   
4                    Lamppost_v1       232.0    0.364944  7.334941e-01   
17                          Body       198.0    4.306316  7.632508e-01   
5                           body        62.0    4.076426  6.961213e-01   
50                     barbwire0        29.0    8.539379  2.202079e+00   
96                        posts0      

In [83]:
# =========================================================
# PART 3) Final fallback: fill SizeX/Y/Z from Unity mean-by-base-name
#         only if SDs are all <= SD_CUTOFF
# =========================================================

SD_CUTOFF = 2.5

# Base-name column for ET (needed to map fixation rows -> report)
# (You already computed still_fix_names["obj_base"] for the still-unmatched fixations)
# We'll join fixation_id -> obj_base, then map obj_base -> mean sizes if SDs are acceptable.

# 1) Build an "acceptable mean size" lookup from the report
#    Keep only base names that exist in Unity AND have SDs all <= cutoff
acceptable_size_rows = unmatched_unity_size_report.dropna(subset=["unity_rows"]).copy()

acceptable_size_rows["sd_ok"] = (
    (acceptable_size_rows["sizex_sd"].fillna(0) <= SD_CUTOFF) &
    (acceptable_size_rows["sizey_sd"].fillna(0) <= SD_CUTOFF) &
    (acceptable_size_rows["sizez_sd"].fillna(0) <= SD_CUTOFF)
)

acceptable_size_lookup = acceptable_size_rows.loc[
    acceptable_size_rows["sd_ok"],
    ["obj_base", "sizex_mean", "sizey_mean", "sizez_mean",
     "sizex_sd", "sizey_sd", "sizez_sd", "unity_rows"]
].copy()

# 2) Attach acceptable mean sizes to still-unmatched fixation IDs (via obj_base)
still_fix_fill = still_fix_names.merge(
    acceptable_size_lookup,
    on="obj_base",
    how="left"
)

# 3) Merge fixation-level fill values onto et
et = et.merge(
    still_fix_fill[["fixation_id", "sizex_mean", "sizey_mean", "sizez_mean"]],
    on="fixation_id",
    how="left"
)

# 4) Fill SizeX/Y/Z ONLY for:
#    - fixation rows
#    - still_no_match fixations
#    - SizeX/Y/Z currently missing
#    - AND we have acceptable mean sizes (sizex_mean notna)
fix_rows = et["fixation_id"].notna()
still_unmatched_rows = fix_rows & et["fixation_id"].isin(still_no_match_fix_ids)

fill3_cond = still_unmatched_rows & et["sizex_mean"].notna()

# Track before -> after counts
sx_was_na = et["SizeX"].isna()
sy_was_na = et["SizeY"].isna()
sz_was_na = et["SizeZ"].isna()

et.loc[fill3_cond & et["SizeX"].isna(), "SizeX"] = et.loc[fill3_cond & et["SizeX"].isna(), "sizex_mean"]
et.loc[fill3_cond & et["SizeY"].isna(), "SizeY"] = et.loc[fill3_cond & et["SizeY"].isna(), "sizey_mean"]
et.loc[fill3_cond & et["SizeZ"].isna(), "SizeZ"] = et.loc[fill3_cond & et["SizeZ"].isna(), "sizez_mean"]

filled3_sx = int((fill3_cond & sx_was_na & et["SizeX"].notna()).sum())
filled3_sy = int((fill3_cond & sy_was_na & et["SizeY"].notna()).sum())
filled3_sz = int((fill3_cond & sz_was_na & et["SizeZ"].notna()).sum())

print("\n---- Mean-size fallback fill (SD cutoff) ----")
print(f"SD_CUTOFF: {SD_CUTOFF}")
print(f"Eligible still-unmatched fixation rows with acceptable mean sizes: {int(fill3_cond.sum())}")
print(f"Rows actually filled from means: SizeX {filled3_sx}, SizeY {filled3_sy}, SizeZ {filled3_sz}")

# 5) Report how many fixations are still missing any size after this step
still_missing_after = (
    et.loc[still_unmatched_rows]
      .groupby("fixation_id")[["SizeX", "SizeY", "SizeZ"]]
      .apply(lambda df: df.isna().any().any())
)

n_fix_still_missing = int(still_missing_after.sum())
print(f"Still-unmatched fixations that remain missing at least one size value: {n_fix_still_missing} / {len(still_no_match_fix_ids)}")

# Optional: list the remaining fixation IDs for inspection
remaining_fix_ids = still_missing_after[still_missing_after].index.to_list()
print("\nFirst 50 remaining fixation_ids with no size filled:")
print(remaining_fix_ids[:50])

# Optional cleanup: remove helper mean columns
et.drop(columns=["sizex_mean", "sizey_mean", "sizez_mean"], inplace=True)


---- Mean-size fallback fill (SD cutoff) ----
SD_CUTOFF: 2.5
Eligible still-unmatched fixation rows with acceptable mean sizes: 37023
Rows actually filled from means: SizeX 37023, SizeY 37023, SizeZ 37023
Still-unmatched fixations that remain missing at least one size value: 10717 / 13007

First 50 remaining fixation_ids with no size filled:
[22, 27, 28, 232, 233, 234, 240, 241, 249, 269, 307, 313, 314, 326, 341, 342, 362, 480, 504, 506, 518, 519, 520, 522, 523, 526, 531, 541, 602, 603, 604, 606, 607, 608, 609, 610, 611, 612, 613, 616, 617, 618, 619, 620, 621, 622, 623, 624, 662, 663]


In [84]:
# =========================================================
# FINAL SUMMARY: counts of fixations matched by each step
#   1) safe merge (name+collider safe pairs)
#   2) proximity merge (accepted proximity)
#   3) fallback mean-size merge (SD-filtered base-name means)
#   4) still no size match
#
# Assumptions based on your pipeline:
#   - et has fixation_id (NaN for non-fixation rows)
#   - et has is_safe_match merged by fixation_id
#   - results_df exists from proximity step (one row per attempted fixation)
#   - still_fix_fill exists from fallback step (fixation_id -> obj_base + mean sizes for sd_ok)
#   - SizeX/SizeY/SizeZ exist in et (filled when matched)
# =========================================================

# 0) fixation universe
fix_ids_all = et.loc[et["fixation_id"].notna(), "fixation_id"].dropna().unique()
n_fix_total = len(fix_ids_all)

# 1) safe-merge matches (fixations whose pair was in safe set)
safe_fix_ids = (
    et.loc[et["fixation_id"].notna() & (et["is_safe_match"] == True), "fixation_id"]
      .dropna()
      .unique()
)
safe_fix_set = set(safe_fix_ids)

# 2) proximity-accepted matches (from results_df)
prox_fix_set = set(
    results_df.loc[results_df["accepted"] == True, "fixation_id"].dropna().unique()
)

# 3) fallback aggregation matches:
#    those fixations in still_fix_fill that actually had acceptable means (non-NaN)
fallback_fix_set = set(
    still_fix_fill.loc[still_fix_fill["sizex_mean"].notna(), "fixation_id"].dropna().unique()
)

# Make sets disjoint in the logical pipeline order (safe > proximity > fallback)
prox_only_set = prox_fix_set - safe_fix_set
fallback_only_set = fallback_fix_set - safe_fix_set - prox_fix_set

# 4) still no match (no SizeX/Y/Z filled at all for that fixation)
fix_size_any = (
    et.loc[et["fixation_id"].notna(), ["fixation_id", "SizeX", "SizeY", "SizeZ"]]
      .groupby("fixation_id")[["SizeX", "SizeY", "SizeZ"]]
      .apply(lambda df: df.notna().any().any())
)

no_match_set = set(fix_size_any.index[~fix_size_any].to_list())

# sanity: these are all fixations that exist
all_fix_set = set(fix_ids_all)

# Counts
n_safe = len(safe_fix_set)
n_prox_only = len(prox_only_set)
n_fallback_only = len(fallback_only_set)
n_no_match = len(no_match_set)

# Optional: also compute "matched by any size fill"
matched_any_set = all_fix_set - no_match_set
n_matched_any = len(matched_any_set)

print("\n================ FIXATION MERGE SUMMARY ================")
print(f"Total fixations in ET (valid onset..offset): {n_fix_total}")
print(f"Matched in SAFE step (name+collider safe pairs): {n_safe}")
print(f"Matched in PROXIMITY step (accepted), excluding safe: {n_prox_only}")
print(f"Matched in FALLBACK mean-size step (SD-filtered), excluding prior: {n_fallback_only}")
print(f"Still NO match (SizeX/Y/Z all missing): {n_no_match}")
print("--------------------------------------------------------")
print(f"Matched by any method (has any SizeX/Y/Z): {n_matched_any}")

# Consistency check: safe + prox_only + fallback_only + no_match should equal total
check_sum = n_safe + n_prox_only + n_fallback_only + n_no_match
print("--------------------------------------------------------")
print(f"Check sum (should equal total): {check_sum}")


================ FIXATION MERGE SUMMARY ================
Total fixations in ET (valid onset..offset): 147066
Matched in SAFE step (name+collider safe pairs): 107898
Matched in PROXIMITY step (accepted), excluding safe: 26161
Matched in FALLBACK mean-size step (SD-filtered), excluding prior: 2290
Still NO match (SizeX/Y/Z all missing): 10717
--------------------------------------------------------
Matched by any method (has any SizeX/Y/Z): 136349
--------------------------------------------------------
Check sum (should equal total): 147066


In [85]:
# ---------------------------------------------------------
# Drop helper / intermediate columns and overwrite ET CSV
# ---------------------------------------------------------

cols_to_drop = [
    "CenterX", "CenterY", "CenterZ",
    "is_safe_match",
    "collider_type_norm_fix",
    "collider_type_norm",
    "obj_name_norm",
]

# Drop only columns that actually exist (defensive)
cols_existing = [c for c in cols_to_drop if c in et.columns]
et.drop(columns=cols_existing, inplace=True)

print("Dropped columns:")
print(cols_existing)

# ---------------------------------------------------------
# Overwrite the original combined_dataframe.csv
# ---------------------------------------------------------
out_path = r"C:\Users\Marcel\Desktop\Study Project\VR Data\Data with Turns\combined_dataframe.csv"
et.to_csv(out_path, index=False)

print(f"ET dataframe written to:\n{out_path}")

Dropped columns:
['CenterX', 'CenterY', 'CenterZ', 'is_safe_match', 'collider_type_norm_fix', 'collider_type_norm', 'obj_name_norm']
ET dataframe written to:
C:\Users\Marcel\Desktop\Study Project\VR Data\Data with Turns\combined_dataframe.csv
